# Phase 2 — Jupyter run

Hybrid deepfake detection (EfficientNet-B0 + FFT + Bi-LSTM). Run every code cell from top to bottom.

Put `sdfvd.zip` in the project folder (`DL PROJECT/sdfvd.zip`) before the SDFVD cell. Outputs are written to `DL PROJECT/deepfake_phase2`.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

# Project folder: the one that contains deepfake_hybrid/ and this notebooks/ folder.
HERE = Path.cwd().resolve()
if (HERE / "deepfake_hybrid").is_dir():
    PROJECT_DIR = HERE
elif (HERE.parent / "deepfake_hybrid").is_dir():
    PROJECT_DIR = HERE.parent
else:
    PROJECT_DIR = Path(r"c:\Users\itl\Desktop\DL PROJECT")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

def run(*args):
    """Run a project module with this notebook's Python."""
    subprocess.check_call([sys.executable, "-m", *args], cwd=str(PROJECT_DIR))

print("project:", PROJECT_DIR)
print("python:", sys.executable)

project: C:\Users\itl\Desktop\DL PROJECT
python: C:\Users\itl\anaconda3\python.exe


In [2]:
print("Next cell installs packages into this kernel. It can take a few minutes.")

Next cell installs packages into this kernel. It can take a few minutes.


On this PC, `DRIVE_ROOT` is the project folder and outputs go to `deepfake_phase2/`.

`facenet-pytorch` is installed with `--no-deps` so the existing PyTorch is left in place.

In [3]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "Pillow", "opencv-python", "numpy", "tqdm", "timm", "scikit-learn", "matplotlib", "grad-cam"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "facenet-pytorch", "--no-deps"])
print("packages ready")

packages ready


## 1. Inspect the FaceForensics++ mirror

Prints the features, the official splits, and the category counts. Saves one sample frame and guesses whether it is a full frame or a face crop.

Open `OUTPUT_DIR/inspect/sample_frame.jpg`. If the guess is wrong, set `FACE_INPUT_MODE` in `config.py` to `full_frame` or `face_crop`, then run this cell again.

In [ ]:
run("deepfake_hybrid.data_prep", "--step", "inspect")

## 2. FF++ subset, split, and cached faces

Pools the configured HF splits, keeps original + the four manipulation types, balances real and fake **videos**, splits 70/15/15 by identity, and caches 224x224 faces plus FFT maps.

Counts are whatever this run finds. They are saved under `OUTPUT_DIR/manifests/`.

In [ ]:
run("deepfake_hybrid.data_prep", "--step", "ffpp")

## 3. SDFVD (test only)

Unzips `DRIVE_ROOT/sdfvd.zip`, prints the folder tree, then extracts every 5th frame (cap 60) and runs the MTCNN crop. No training uses these videos.

In [ ]:
run("deepfake_hybrid.data_prep", "--step", "sdfvd")

## 4. c40-style (simulated)

Re-encodes the FF++ **test** frames (before the face crop) with `ffmpeg libx264 -crf 40`, reads them back, and applies the same face step. This is not the official FF++ c40 release.

In [ ]:
run("deepfake_hybrid.data_prep", "--step", "c40")

## 5. Train all four variants and evaluate once

Variants: spatial-only baseline, spatial+frequency, spatial+temporal, full hybrid.

The checkpoint for each variant is the epoch with the highest validation sequence AUC. The test sets (FF++ c23, c40-style (simulated), SDFVD) are scored once at the end. Results land in `OUTPUT_DIR/results/metrics.csv` and `metrics.md`, with ROC curves and confusion matrices beside them.

The first run downloads EfficientNet-B0 ImageNet weights. A later run skips variants that already have a checkpoint and does not score the test set again if `metrics.csv` is already there.

In [ ]:
run("deepfake_hybrid.train", "--all")

## 6. Grad-CAM

Overlays from the full hybrid's EfficientNet-B0 branch: 2 real and 2 fake frames from the FF++ c23 test split, and 2 real and 2 fake frames from SDFVD. Saved under `OUTPUT_DIR/results/gradcam/`.

In [ ]:
run("deepfake_hybrid.explain")